In [1]:
import tensorflow as tf
print(tf.__version__)


2.18.0


In [5]:
!pip install streamlit==1.30.0 pyngrok


In [16]:
%%writefile app.py
import os
import cv2
import numpy as np
import streamlit as st
import tensorflow as tf

# Load the trained model
model = tf.keras.models.load_model('my_model.keras')

# Define disease causes and solutions
disease_info = {
    'Powdery Mildew': {'cause': 'Humidity (Fungal Infection)', 'factor': 'Airborne', 'survival_days': 30, 'solution': 'Use fungicides, improve air circulation', 'treatment_possible': True},
    'Leaf Rust': {'cause': 'Fungal Spores in Soil', 'factor': 'Soil', 'survival_days': 40, 'solution': 'Apply fungicide, remove infected leaves', 'treatment_possible': True},
    'Bacterial Blight': {'cause': 'Water Contamination', 'factor': 'Water', 'survival_days': 20, 'solution': 'Use clean water, apply copper-based bactericide', 'treatment_possible': True},
    'Pest Infestation': {'cause': 'Pesticide Resistance', 'factor': 'Pesticides', 'survival_days': 25, 'solution': 'Introduce natural predators, rotate pesticides', 'treatment_possible': True},
    'Root Rot': {'cause': 'Excess Soil Moisture', 'factor': 'Soil', 'survival_days': 15, 'solution': 'Improve drainage, reduce watering', 'treatment_possible': False},
    'Anthracnose': {'cause': 'Fungal Infection in Warm Weather', 'factor': 'Airborne', 'survival_days': 35, 'solution': 'Apply copper fungicide, remove infected parts', 'treatment_possible': True},
    'Mosaic Virus': {'cause': 'Viral Infection through Insects', 'factor': 'Insects', 'survival_days': 50, 'solution': 'Use resistant plant varieties, control insect vectors', 'treatment_possible': False},
    'Late Blight': {'cause': 'Waterborne Fungal Spores', 'factor': 'Water', 'survival_days': 10, 'solution': 'Remove infected plants, apply fungicide', 'treatment_possible': False},
    'Downy Mildew': {'cause': 'High Humidity and Poor Ventilation', 'factor': 'Airborne', 'survival_days': 28, 'solution': 'Improve airflow, apply appropriate fungicides', 'treatment_possible': True},
    'Wilt Disease': {'cause': 'Soilborne Fungal Infection', 'factor': 'Soil', 'survival_days': 45, 'solution': 'Use disease-resistant varieties, rotate crops', 'treatment_possible': True}
}

# Initialize chat history
if "history" not in st.session_state:
    st.session_state.history = []

# Function to predict disease and provide additional details
def predict_disease(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (224, 224))
    img = np.expand_dims(img, axis=0) / 255.0

    prediction = model.predict(img)
    predicted_class = np.argmax(prediction)
    class_labels = list(disease_info.keys())

    if predicted_class >= len(class_labels):
        return {
            'Disease': 'Unknown Disease',
            'Cause': 'Not in database',
            'Factor': 'Unknown',
            'Estimated Survival': 'Unknown',
            'Solution': 'Consult an agricultural expert',
            'Treatment Possible': 'Unknown',
            'Recommendation': 'Further diagnosis needed'
        }

    disease_name = class_labels[predicted_class]
    info = disease_info[disease_name]

    recommendation = 'Remove the plant' if not info['treatment_possible'] else 'Apply treatment'

    return {
        'Disease': disease_name,
        'Cause': info['cause'],
        'Factor': info['factor'],
        'Estimated Survival': f"{info['survival_days']} days",
        'Solution': info['solution'],
        'Treatment Possible': 'Yes' if info['treatment_possible'] else 'No',
        'Recommendation': recommendation
    }

# Streamlit UI
st.set_page_config(page_title="Plant Disease Detector", page_icon="🌿", layout="wide")

# Add logo
st.image("/content/logo.png", width=150)  # Make sure "logo.png" is in the same folder

st.markdown("<h1 style='text-align: center; color: green;'>🌿 Plant Disease Prediction System 🌿</h1>", unsafe_allow_html=True)
st.sidebar.title('📁 Upload Image')
st.sidebar.markdown('Upload a plant leaf image to detect disease and get recommendations.')

# Upload file
uploaded_file = st.sidebar.file_uploader("Choose an image", type=['jpg', 'png', 'jpeg'])

if uploaded_file is not None:
    image_path = os.path.join("/content/temp.jpg")
    with open(image_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    st.image(uploaded_file, caption='Uploaded Image', use_column_width=True)

    if st.sidebar.button("🔍 Predict Disease"):
        result = predict_disease(image_path)

        # Save chat history with disease name as the title
        st.session_state.history.append(result)

        st.markdown("---")
        st.markdown(f"<h2 style='color: red;'>🦠 Detected Disease: {result['Disease']}</h2>", unsafe_allow_html=True)
        st.write(f"**Cause:** {result['Cause']}")
        st.write(f"**Factor:** {result['Factor']}")
        st.write(f"**Estimated Survival:** {result['Estimated Survival']}")
        st.write(f"**Solution:** {result['Solution']}")
        st.write(f"**Treatment Possible:** {result['Treatment Possible']}")
        st.write(f"**Recommendation:** {result['Recommendation']}")
        st.markdown("---")

# Display chat history with disease names as section titles
if st.session_state.history:
    st.sidebar.markdown("### Chat History")
    for entry in st.session_state.history:
        with st.sidebar.expander(f"📄 {entry['Disease']}"):
            st.write(f"**Cause:** {entry['Cause']}")
            st.write(f"**Solution:** {entry['Solution']}")
            st.write(f"**Recommendation:** {entry['Recommendation']}")


Overwriting app.py


In [19]:

!ngrok config add-authtoken 2tkdZOaCoq6qTph9qmQMWVNnwjO_5DFGQJUckT9Ff2FXTak3P




Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [20]:
!streamlit run app.py &>/content/logs.txt &





In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8501).public_url
print("Streamlit is running at:", public_url)